# Important: In your agent.py, you need to add this to the top of the file:

```
import sys, glob
sys.path.append(glob.glob('/kaggle/input/**/cg-lib', recursive=True)[0])
```

In [40]:
%%writefile battle_simulator.py
"""
CABT head-to-head battle simulator for local/debug evaluation.

Use case:
  python battle_simulator.py \
      --agent-a deck1.py \
      --agent-b deck2.py \
      --deck-a deck1.csv \
      --deck-b deck2.csv \
      --matches 100

Debug IDs / action metadata:
  DEBUG_AGENT=1 python battle_simulator.py \
      --agent-a deck1.py \
      --agent-b deck2.py \
      --deck-a deck1.csv \
      --deck-b deck2.csv \
      --matches 1 \
      --debug \
      --trace-dir traces \
      --dump-card-ids 1,2,3

This script is intended to run inside an environment where CABT's `cg` package is
available. It uses the low-level CABT game API:
  - battle_start(deck0, deck1)
  - battle_select(select_list)
  - battle_finish()
  - visualize_data()

The script is deliberately defensive because CABT dataclass/dict fields can vary
slightly across versions. It tries both object attributes and dictionary keys.
"""

from __future__ import annotations

import argparse
import csv
import importlib.util
import inspect
import json
import os
import random
import re
import sys
import time
import traceback
from contextlib import contextmanager
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from types import ModuleType
from typing import Any, Callable, Iterable

import glob

sys.path.append(glob.glob('/kaggle/input/**/cg-lib', recursive=True)[0])

try:
    from cg.game import battle_start, battle_select, battle_finish, visualize_data
except Exception as e:  # pragma: no cover - only available in CABT runtime
    raise RuntimeError(
        "Could not import CABT game API. Run this inside the CABT/Kaggle environment "
        "where `cg.game` is available."
    ) from e

try:
    from cg.api import all_card_data, all_attack
except Exception:  # pragma: no cover - depends on CABT runtime
    all_card_data = None
    all_attack = None


# ---------------------------------------------------------------------------
# Generic helpers for CABT dict/dataclass access
# ---------------------------------------------------------------------------

_MISSING = object()


def get_any(obj: Any, *keys: str, default: Any = None) -> Any:
    """Return the first matching attr/key from an object or dict."""
    if obj is None:
        return default
    for key in keys:
        if isinstance(obj, dict) and key in obj:
            return obj[key]
        if hasattr(obj, key):
            return getattr(obj, key)
    return default


def get_path(obj: Any, path: str, default: Any = None) -> Any:
    cur = obj
    for part in path.split("."):
        if cur is None:
            return default
        if part.isdigit():
            idx = int(part)
            try:
                cur = cur[idx]
            except Exception:
                return default
        else:
            cur = get_any(cur, part, default=_MISSING)
            if cur is _MISSING:
                return default
    return cur


def to_plain(obj: Any, max_depth: int = 6) -> Any:
    """Best-effort JSON-ish conversion for debug traces."""
    if max_depth <= 0:
        return repr(obj)
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, dict):
        return {str(k): to_plain(v, max_depth - 1) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_plain(x, max_depth - 1) for x in obj]
    if hasattr(obj, "__dict__"):
        out = {}
        for k, v in vars(obj).items():
            if k.startswith("_"):
                continue
            out[k] = to_plain(v, max_depth - 1)
        return out
    return repr(obj)


def enum_name(x: Any) -> str:
    """Readable name for enum-ish values."""
    if x is None:
        return "None"
    if hasattr(x, "name"):
        return str(x.name)
    return str(x)


def card_id(card_or_pokemon: Any) -> int | None:
    val = get_any(card_or_pokemon, "id", "cardId", "card_id", default=None)
    try:
        return None if val is None else int(val)
    except Exception:
        return None


# ---------------------------------------------------------------------------
# Metadata dump helpers
# ---------------------------------------------------------------------------


def object_public_attrs(obj: Any) -> dict[str, Any]:
    """Return simple public attrs for debugging card/attack dataclasses."""
    attrs: dict[str, Any] = {}
    if isinstance(obj, dict):
        iterator = obj.items()
    else:
        if hasattr(obj, "__dict__"):
            iterator = vars(obj).items()
        else:
            names = [n for n in dir(obj) if not n.startswith("_")]
            tmp = []
            for n in names:
                try:
                    v = getattr(obj, n)
                except Exception:
                    continue
                if not callable(v):
                    tmp.append((n, v))
            iterator = tmp
    for k, v in iterator:
        if str(k).startswith("_"):
            continue
        if isinstance(v, (str, int, float, bool)) or v is None:
            attrs[str(k)] = v
        elif isinstance(v, (list, tuple)):
            attrs[str(k)] = [to_plain(x, 2) for x in v[:8]]
        else:
            attrs[str(k)] = repr(v)
    return attrs


def build_card_name_table() -> dict[int, str]:
    out: dict[int, str] = {}
    if all_card_data is None:
        return out
    try:
        for c in all_card_data():
            cid = get_any(c, "cardId", "id", default=None)
            name = get_any(c, "name", "cardName", default="")
            if cid is not None:
                out[int(cid)] = str(name)
    except Exception:
        pass
    return out


def card_name(cid: int | None, name_table: dict[int, str]) -> str:
    if cid is None:
        return ""
    return name_table.get(cid, f"card:{cid}")


def dump_card_metadata(card_ids: list[int], out_dir: Path, stdout: bool = True) -> None:
    if all_card_data is None:
        print("[meta] all_card_data() unavailable", file=sys.stderr)
        return
    try:
        cards = list(all_card_data())
    except Exception as e:
        print(f"[meta] all_card_data() failed: {e!r}", file=sys.stderr)
        return

    wanted = set(card_ids)
    rows = []
    for c in cards:
        cid = get_any(c, "cardId", "id", default=None)
        try:
            cid_int = int(cid)
        except Exception:
            continue
        if cid_int in wanted:
            row = object_public_attrs(c)
            row["_cardId"] = cid_int
            rows.append(row)

    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / "card_metadata_debug.json"
    path.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")

    if stdout:
        print(f"[meta] wrote {path}")
        for row in rows:
            print("[card]", json.dumps(row, ensure_ascii=False)[:1200])


def dump_attack_metadata(
    out_dir: Path,
    interesting_names: list[str] | None = None,
    stdout: bool = True,
) -> None:
    if all_attack is None:
        print("[meta] all_attack() unavailable", file=sys.stderr)
        return
    try:
        attacks = list(all_attack())
    except Exception as e:
        print(f"[meta] all_attack() failed: {e!r}", file=sys.stderr)
        return

    out_dir.mkdir(parents=True, exist_ok=True)
    rows = [object_public_attrs(a) for a in attacks]
    path = out_dir / "all_attack_metadata_debug.json"
    path.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")
    if stdout:
        print(f"[meta] wrote {path} with {len(rows)} attacks/abilities")

    names = interesting_names
    needles = [n.lower().replace("-", " ") for n in names]

    filtered = []
    for row in rows:
        joined = " ".join(str(v) for v in row.values()).lower().replace("-", " ")
        if any(n in joined for n in needles):
            filtered.append(row)

    fpath = out_dir / "interesting_attack_metadata_debug.json"
    fpath.write_text(json.dumps(filtered, indent=2, ensure_ascii=False), encoding="utf-8")
    if stdout:
        print(f"[meta] wrote {fpath} with {len(filtered)} filtered rows")
        for row in filtered[:80]:
            print("[attack]", json.dumps(row, ensure_ascii=False)[:1600])


# ---------------------------------------------------------------------------
# Agent import and deck loading
# ---------------------------------------------------------------------------


@dataclass
class LoadedAgent:
    label: str
    path: Path
    module: ModuleType
    fn: Callable[[dict], list[int]]
    deck: list[int]


@contextmanager
def staged_deck_csv(agent_path: Path, deck: list[int] | None):
    """Temporarily materialize deck.csv next to an agent during import.

    Many scaffolded CABT agents execute this at module import time:

        file_path = "deck.csv"
        if not os.path.exists(file_path):
            file_path = "/kaggle_simulations/agent/deck.csv"
        with open(file_path) as f: ...

    That means the simulator cannot simply import the module and inject the deck
    later. If we know the intended deck, we write it as `deck.csv` only for the
    duration of the import, then restore whatever was there before.
    """
    target = agent_path.parent / "deck.csv"
    if deck is None:
        yield
        return

    existed = target.exists()
    old_bytes = target.read_bytes() if existed else None
    target.write_text("\n".join(str(x) for x in deck) + "\n", encoding="utf-8")
    try:
        yield
    finally:
        if existed and old_bytes is not None:
            target.write_bytes(old_bytes)
        else:
            try:
                target.unlink()
            except FileNotFoundError:
                pass


def import_agent_module(path: Path, label: str, staged_deck: list[int] | None = None) -> ModuleType:
    """Import an agent file using its own directory as cwd during import.

    This matters because many CABT agents load `deck.csv` at import time from cwd.
    If `staged_deck` is provided, the simulator temporarily writes that deck as
    `deck.csv` next to the agent before executing the module.
    """
    path = path.resolve()
    if not path.exists():
        raise FileNotFoundError(path)

    mod_name = f"cabt_agent_{label}_{abs(hash(str(path))) & 0xFFFFFFFF:x}_{int(time.time() * 1e6) % 1000000:x}"
    spec = importlib.util.spec_from_file_location(mod_name, str(path))
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not import {path}")

    old_cwd = os.getcwd()
    old_sys_path = list(sys.path)
    try:
        with staged_deck_csv(path, staged_deck):
            os.chdir(str(path.parent))
            sys.path.insert(0, str(path.parent))
            module = importlib.util.module_from_spec(spec)
            sys.modules[mod_name] = module
            spec.loader.exec_module(module)
            return module
    finally:
        os.chdir(old_cwd)
        sys.path[:] = old_sys_path


def load_deck_file(deck_path: Path) -> list[int]:
    raw = deck_path.read_text(encoding="utf-8").splitlines()
    ids: list[int] = []
    for line in raw:
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        # Accept one ID per line, or CSV-like first column.
        first = re.split(r"[,\s]+", s)[0]
        if first:
            ids.append(int(first))
    if len(ids) != 60:
        raise ValueError(f"{deck_path} contained {len(ids)} IDs; CABT requires exactly 60")
    return ids


def parse_decklist_from_agent_source(agent_path: Path) -> list[int] | None:
    """Best-effort parser for scaffold comments like `Riolu = 677  # ×3`.

    Public CABT scaffold agents often include a complete decklist as Python
    constants plus inline copy-count comments, but still try to read deck.csv at
    import time. This parser extracts those constants before import so we can
    stage a valid deck.csv and avoid FileNotFoundError.
    """
    try:
        text = agent_path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        text = agent_path.read_text(encoding="utf-8-sig")

    # Only parse the explicit Decklist block if present; otherwise fall back to
    # scanning all constant assignments with copy-count comments.
    if "# Decklist" in text:
        text_to_scan = text.split("# Decklist", 1)[1]
        # Stop at first class/function definition after the decklist block.
        text_to_scan = re.split(r"\n(?:class|def)\s+", text_to_scan, maxsplit=1)[0]
    else:
        text_to_scan = text

    deck: list[int] = []
    # Matches examples:
    #   Mega_Lucario_ex = 678  # ×4
    pat = re.compile(
        r"^\s*[A-Za-z_]\w*\s*=\s*(\d+)\s*#.*?(?:×|x|X)\s*(\d+)\b",
        re.MULTILINE,
    )
    for m in pat.finditer(text_to_scan):
        card_id_i = int(m.group(1))
        count_i = int(m.group(2))
        deck.extend([card_id_i] * count_i)

    if len(deck) == 60:
        return deck
    return None


def resolve_deck_before_import(agent_path: Path, explicit_deck: str | None) -> tuple[list[int] | None, str]:
    """Return a deck we can stage before import, plus a human-readable source."""
    if explicit_deck:
        return load_deck_file(Path(explicit_deck)), f"explicit deck file {explicit_deck}"

    sidecar = agent_path.parent / "deck.csv"
    if sidecar.exists():
        return load_deck_file(sidecar), f"sidecar {sidecar}"

    parsed = parse_decklist_from_agent_source(agent_path)
    if parsed is not None:
        return parsed, "parsed # Decklist comments from agent source"

    return None, "unresolved before import"


def infer_deck(module: ModuleType, agent_path: Path, explicit_deck: str | None, preimport_deck: list[int] | None = None) -> list[int]:
    if preimport_deck is not None:
        return [int(x) for x in preimport_deck]

    if explicit_deck:
        return load_deck_file(Path(explicit_deck))

    sidecar = agent_path.parent / "deck.csv"
    if sidecar.exists():
        return load_deck_file(sidecar)

    for attr in ("my_deck", "FALLBACK_DECK", "deck"):
        if hasattr(module, attr):
            value = list(getattr(module, attr))
            deck = [int(x) for x in value]
            if len(deck) == 60:
                return deck

    # Last resort: some agents return deck when select/current are None.
    # This may fail if the agent assumes exact CABT observation schema.
    probe_obs = {"logs": [], "current": None, "select": None}
    try:
        deck = module.agent(probe_obs)
        if isinstance(deck, list) and len(deck) == 60 and all(isinstance(x, int) for x in deck):
            return deck
    except Exception:
        pass

    raise ValueError(
        f"Could not infer deck for {agent_path}. Pass --deck-a/--deck-b explicitly, "
        "put deck.csv next to the agent, or expose module.my_deck/FALLBACK_DECK."
    )


def load_agent(path_str: str, label: str, explicit_deck: str | None) -> LoadedAgent:
    path = Path(path_str).resolve()
    preimport_deck, deck_source = resolve_deck_before_import(path, explicit_deck)
    if preimport_deck is not None:
        print(f"[deck] {label}: staging deck.csv from {deck_source}")
    else:
        print(
            f"[deck] {label}: no deck resolved before import; import may fail if the agent reads deck.csv at module import time"
        )

    module = import_agent_module(path, label, staged_deck=preimport_deck)
    if not hasattr(module, "agent"):
        raise AttributeError(f"{path} does not define an agent(obs_dict) function")
    fn = getattr(module, "agent")
    deck = infer_deck(module, path, explicit_deck, preimport_deck=preimport_deck)
    return LoadedAgent(label=label, path=path, module=module, fn=fn, deck=deck)


# ---------------------------------------------------------------------------
# Observation / selection debug tracing
# ---------------------------------------------------------------------------


def get_select(obs: Any) -> Any:
    return get_path(obs, "select", default=None)


def get_current(obs: Any) -> Any:
    return get_path(obs, "current", default=None)


def get_players(obs: Any) -> list[Any]:
    players = get_path(obs, "current.players", default=[])
    return list(players or [])


def get_your_index(obs: Any) -> int:
    for path in (
        "current.yourIndex",
        "current.your_index",
        "yourIndex",
        "your_index",
        "playerIndex",
        "player_index",
    ):
        val = get_path(obs, path, default=None)
        if val is not None:
            try:
                return int(val)
            except Exception:
                pass
    # If CABT ever returns no perspective, assume current player 0.
    return 0


def get_select_bounds(obs: Any) -> tuple[int, int, int]:
    select = get_select(obs)
    options = get_any(select, "option", "options", default=[])
    min_count = get_any(select, "minCount", "min_count", default=1)
    max_count = get_any(select, "maxCount", "max_count", default=min_count)
    try:
        min_count = int(min_count)
        max_count = int(max_count)
    except Exception:
        min_count = 1
        max_count = 1
    return len(options or []), min_count, max_count


def summarize_card_reference(option: Any, obs: Any, name_table: dict[int, str]) -> dict[str, Any]:
    """Best-effort: identify the card/Pokemon object referenced by an option."""
    out: dict[str, Any] = {}
    area = get_any(option, "area", default=None)
    index = get_any(option, "index", default=None)
    player_index = get_any(option, "playerIndex", "player_index", default=get_your_index(obs))
    out["area"] = enum_name(area)
    out["index"] = index
    out["playerIndex"] = player_index

    # Option may already contain card-like fields in some CABT versions.
    direct_card = get_any(option, "card", "pokemon", default=None)
    cid = card_id(direct_card)
    if cid is not None:
        out["cardId"] = cid
        out["cardName"] = card_name(cid, name_table)
        return out

    # Try to follow area/index/playerIndex in the observation.
    try:
        pi = int(player_index) if player_index is not None else get_your_index(obs)
        idx = int(index) if index is not None else None
    except Exception:
        pi, idx = get_your_index(obs), None

    if idx is None:
        return out

    area_s = enum_name(area).upper()
    players = get_players(obs)
    ps = players[pi] if 0 <= pi < len(players) else None

    candidate = None
    if "HAND" in area_s:
        candidate = get_any(ps, "hand", default=[])[idx]
    elif "DISCARD" in area_s:
        candidate = get_any(ps, "discard", default=[])[idx]
    elif "ACTIVE" in area_s:
        candidate = get_any(ps, "active", default=[])[idx]
    elif "BENCH" in area_s:
        candidate = get_any(ps, "bench", default=[])[idx]
    elif "PRIZE" in area_s:
        candidate = get_any(ps, "prize", default=[])[idx]
    elif "STADIUM" in area_s:
        candidate = get_path(obs, "current.stadium", default=[])[idx]
    elif "LOOKING" in area_s:
        candidate = get_path(obs, "current.looking", default=[])[idx]
    elif "DECK" in area_s:
        select_deck = get_path(obs, "select.deck", default=[])
        candidate = select_deck[idx] if idx < len(select_deck) else None

    cid = card_id(candidate)
    if cid is not None:
        out["cardId"] = cid
        out["cardName"] = card_name(cid, name_table)
    return out


def summarize_option(i: int, option: Any, obs: Any, name_table: dict[int, str]) -> dict[str, Any]:
    row = {
        "i": i,
        "type": enum_name(get_any(option, "type", default=None)),
        "attackId": get_any(option, "attackId", "attack_id", default=None),
        "abilityId": get_any(option, "abilityId", "ability_id", default=None),
        "number": get_any(option, "number", default=None),
        "inPlayArea": enum_name(get_any(option, "inPlayArea", "in_play_area", default=None)),
        "inPlayIndex": get_any(option, "inPlayIndex", "in_play_index", default=None),
    }
    row.update(summarize_card_reference(option, obs, name_table))
    return row


def summarize_select(obs: Any, name_table: dict[int, str]) -> dict[str, Any]:
    select = get_select(obs)
    if select is None:
        return {"select": None}
    options = list(get_any(select, "option", "options", default=[]) or [])
    effect = get_any(select, "effect", default=None)
    context_card = get_any(select, "contextCard", "context_card", default=None)
    return {
        "context": enum_name(get_any(select, "context", default=None)),
        "minCount": get_any(select, "minCount", "min_count", default=None),
        "maxCount": get_any(select, "maxCount", "max_count", default=None),
        "remainDamageCounter": get_any(select, "remainDamageCounter", "remain_damage_counter", default=None),
        "effectId": card_id(effect),
        "effectName": card_name(card_id(effect), name_table),
        "contextCardId": card_id(context_card),
        "contextCardName": card_name(card_id(context_card), name_table),
        "options": [summarize_option(i, o, obs, name_table) for i, o in enumerate(options)],
    }


def summarize_logs(obs: Any, name_table: dict[int, str]) -> list[dict[str, Any]]:
    logs = get_path(obs, "logs", default=[]) or []
    out = []
    for log in logs:
        row = object_public_attrs(log)
        for k in ("card", "pokemon", "effect", "contextCard"):
            obj = get_any(log, k, default=None)
            cid = card_id(obj)
            if cid is not None:
                row[f"{k}Id"] = cid
                row[f"{k}Name"] = card_name(cid, name_table)
        if "type" in row:
            row["type"] = enum_name(row["type"])
        out.append(row)
    return out


def summarize_state(obs: Any, name_table: dict[int, str]) -> dict[str, Any]:
    cur = get_current(obs)
    if cur is None:
        return {}
    players = get_any(cur, "players", default=[]) or []
    p_rows = []
    for pi, p in enumerate(players):
        active = get_any(p, "active", default=[]) or []
        bench = get_any(p, "bench", default=[]) or []
        p_rows.append(
            {
                "player": pi,
                "prizes": len(get_any(p, "prize", default=[]) or []),
                "handCount": get_any(p, "handCount", "hand_count", default=None),
                "deckCount": get_any(p, "deckCount", "deck_count", default=None),
                "active": [
                    {
                        "id": card_id(x),
                        "name": card_name(card_id(x), name_table),
                        "hp": get_any(x, "hp", default=None),
                    }
                    for x in active
                    if x is not None
                ],
                "bench": [
                    {
                        "id": card_id(x),
                        "name": card_name(card_id(x), name_table),
                        "hp": get_any(x, "hp", default=None),
                    }
                    for x in bench
                    if x is not None
                ],
            }
        )
    return {
        "turn": get_any(cur, "turn", default=None),
        "yourIndex": get_any(cur, "yourIndex", "your_index", default=None),
        "firstPlayer": get_any(cur, "firstPlayer", "first_player", default=None),
        "stadium": [card_name(card_id(x), name_table) for x in (get_any(cur, "stadium", default=[]) or [])],
        "players": p_rows,
    }


# ---------------------------------------------------------------------------
# Selection validation and winner detection
# ---------------------------------------------------------------------------


def legal_fallback_selection(obs: Any, rng: random.Random) -> list[int]:
    n_options, min_count, max_count = get_select_bounds(obs)
    if n_options <= 0:
        return []
    # CABT multi-select prompts can reject semantically invalid combinations even
    # when every raw index is in range.  The safest generic fallback is therefore
    # the *minimum* required number of choices, not maxCount.  If minCount == 0,
    # selecting nothing is usually safer than arbitrarily selecting option 0.
    k = max(0, min(min_count, n_options))
    return list(range(k))


def validate_selection(selection: Any, obs: Any) -> tuple[bool, str]:
    n_options, min_count, max_count = get_select_bounds(obs)
    if not isinstance(selection, list):
        return False, f"selection is {type(selection).__name__}, expected list[int]"
    if not all(isinstance(x, int) for x in selection):
        return False, f"selection contains non-int values: {selection!r}"
    if len(selection) < min_count or len(selection) > max_count:
        return False, f"selection length {len(selection)} outside [{min_count}, {max_count}]"
    if len(set(selection)) != len(selection):
        return False, f"selection contains duplicates: {selection!r}"
    bad = [x for x in selection if x < 0 or x >= n_options]
    if bad:
        return False, f"selection indices out of range 0..{n_options - 1}: {bad}"
    return True, "ok"


def prize_lengths(obs: Any) -> list[int | None]:
    """Return visible prize-list lengths for both players when available.

    CABT can expose [0, 0] before setup/prize initialization. That is not
    a terminal state; it only means prizes have not been populated in the
    observation yet.
    """
    players = get_players(obs)
    out: list[int | None] = []
    for p in players[:2]:
        prizes = get_any(p, "prize", default=None)
        out.append(None if prizes is None else len(prizes))
    while len(out) < 2:
        out.append(None)
    return out


def prizes_are_initialized(obs: Any) -> bool:
    """True once CABT appears to have populated real prize cards.

    At battle_start/setup CABT may show both prize arrays as empty. We only
    allow prize-count win detection after seeing both players with nonzero
    prizes at least once, or after the state has advanced into an actual turn.
    """
    lengths = prize_lengths(obs)
    if all(x is not None and x > 0 for x in lengths):
        return True
    turn = get_path(obs, "current.turn", default=None)
    try:
        # A real game turn means setup has completed in the versions of CABT
        # used for the competition. Keep this as a secondary signal only.
        return int(turn) > 0
    except Exception:
        return False




def _zone_cards(player: Any, *zone_names: str) -> list[Any]:
    """Return non-None cards/Pokémon from one or more player zones."""
    cards: list[Any] = []
    for zone_name in zone_names:
        zone = get_any(player, zone_name, default=[])
        if zone is None:
            continue
        if not isinstance(zone, (list, tuple)):
            zone = [zone]
        for card in zone:
            if card is not None:
                cards.append(card)
    return cards


def player_has_any_pokemon_in_play(player: Any) -> bool:
    """True if active or bench visibly contains at least one Pokémon object/card.

    During setup CABT can temporarily show empty/hidden zones, so callers should
    only use this for terminal inference after prize initialization / real turns.
    """
    return len(_zone_cards(player, "active", "bench")) > 0


def no_pokemon_terminal(obs: Any, allow_terminal_fallback: bool = True) -> tuple[bool, int | None, str]:
    """Detect the standard Pokémon loss condition: no Pokémon in play.

    CABT sometimes leaves a forced select prompt with minCount > 0 but no
    options after a player's final Pokémon is discarded.  In that state the
    match is over, but there may be no explicit winner field yet.
    """
    if not allow_terminal_fallback:
        return False, None, "not finished"
    players = get_players(obs)
    if len(players) < 2:
        return False, None, "not finished"
    try:
        turn = int(get_path(obs, "current.turn", default=0) or 0)
    except Exception:
        turn = 0
    # Guard against setup-only empty zones.  Use only once a real turn has
    # happened and prize initialization has been observed by the caller.
    if turn <= 0:
        return False, None, "not finished"

    p0_has = player_has_any_pokemon_in_play(players[0])
    p1_has = player_has_any_pokemon_in_play(players[1])
    if not p0_has and p1_has:
        return True, 1, "player 0 has no Pokémon in play"
    if not p1_has and p0_has:
        return True, 0, "player 1 has no Pokémon in play"
    if not p0_has and not p1_has:
        return True, None, "both players have no Pokémon in play"
    return False, None, "not finished"

def infer_winner(obs: Any, allow_prize_fallback: bool = True) -> tuple[bool, int | None, str]:
    """Return (finished, winner_index_or_None, reason).

    CABT versions differ in field names, so this checks many likely paths.
    `allow_prize_fallback` must be False during setup, because some CABT
    observations show both players with zero prize cards before prizes have
    actually been initialized.
    """
    # Direct winner-like fields.
    for path in (
        "current.winner",
        "current.winnerIndex",
        "current.winner_index",
        "current.winPlayer",
        "current.win_player",
        "current.result.winner",
        "current.result.winnerIndex",
        "winner",
        "winnerIndex",
    ):
        val = get_path(obs, path, default=None)
        if val is not None:
            try:
                winner = int(val)
                if winner in (0, 1):
                    return True, winner, f"winner field {path}={winner}"
            except Exception:
                # Some APIs store string names.
                s = str(val).lower()
                if "player0" in s or "player 0" in s or s == "0":
                    return True, 0, f"winner field {path}={val!r}"
                if "player1" in s or "player 1" in s or s == "1":
                    return True, 1, f"winner field {path}={val!r}"

    # Finished/terminal flag plus loser fields.
    terminal = False
    for path in (
        "current.isFinished",
        "current.finished",
        "current.gameEnd",
        "current.game_end",
        "current.isGameEnd",
        "current.is_game_end",
        "done",
        "terminal",
    ):
        val = get_path(obs, path, default=None)
        if val is not None and bool(val):
            terminal = True
            break
    if terminal:
        for path in ("current.loser", "current.loserIndex", "current.loser_index"):
            val = get_path(obs, path, default=None)
            if val is not None:
                try:
                    loser = int(val)
                    if loser in (0, 1):
                        return True, 1 - loser, f"terminal + loser field {path}={loser}"
                except Exception:
                    pass
        return True, None, "terminal flag but no winner field found"

    # Pokémon-in-play terminal fallback: if a player has no Active/Bench
    # Pokémon after setup has initialized, they lose. This handles CABT states
    # where a forced prompt has minCount > 0 but no selectable options after the
    # last Pokémon was Knocked Out.
    no_pkm_finished, no_pkm_winner, no_pkm_reason = no_pokemon_terminal(
        obs, allow_terminal_fallback=allow_prize_fallback
    )
    if no_pkm_finished:
        return True, no_pkm_winner, no_pkm_reason

    # Prize-based terminal fallback: a player with zero prizes has won.
    # Guard this carefully: right after battle_start / during setup, CABT may
    # expose both prize arrays as empty even though the battle has not begun.
    if allow_prize_fallback:
        lengths = prize_lengths(obs)
        if lengths[0] == 0 and (lengths[1] is None or lengths[1] > 0):
            return True, 0, "player 0 has zero prize cards"
        if lengths[1] == 0 and (lengths[0] is None or lengths[0] > 0):
            return True, 1, "player 1 has zero prize cards"
        if lengths[0] == 0 and lengths[1] == 0:
            # Ambiguous all-zero prize state. This is usually setup/init, not a
            # double terminal. Require an explicit winner field instead.
            return False, None, "not finished: both prize arrays are zero/ambiguous"

    # Avoid inferring losses from hidden face-down setup Pokémon. CABT may
    # represent an unrevealed active as [None], which is not the same as
    # having no Pokémon in play. Rely on explicit winner/game-result fields
    # or guarded zero-prize fallback instead.

    return False, None, "not finished"


# ---------------------------------------------------------------------------
# Match runner
# ---------------------------------------------------------------------------


@dataclass
class MatchResult:
    match_index: int
    seat_a: int
    seat_b: int
    winner_seat: int | None
    winner_label: str | None
    steps: int
    turns: int | None
    reason: str
    invalid_a: int
    invalid_b: int
    agent_exceptions_a: int = 0
    agent_exceptions_b: int = 0
    exception_source: str | None = None
    exception_message: str | None = None
    exception: str | None = None


class JsonlTraceWriter:
    def __init__(self, path: Path | None):
        self.path = path
        self.fp = None
        if path is not None:
            path.parent.mkdir(parents=True, exist_ok=True)
            self.fp = path.open("w", encoding="utf-8")

    def write(self, row: dict[str, Any]) -> None:
        if self.fp is not None:
            self.fp.write(json.dumps(row, ensure_ascii=False) + "\n")

    def close(self) -> None:
        if self.fp is not None:
            self.fp.close()


def call_agent(agent: LoadedAgent, obs: Any) -> list[int]:
    return agent.fn(obs)


def run_match(
    match_index: int,
    agent_a: LoadedAgent,
    agent_b: LoadedAgent,
    seat_a: int,
    debug: bool,
    trace_dir: Path | None,
    max_steps: int,
    rng: random.Random,
    name_table: dict[int, str],
    save_visualize: bool,
    on_invalid_selection: str = "loss",
) -> MatchResult:
    """Run one battle. seat_a is 0 or 1; agent_b gets the other seat."""
    agents_by_seat = [None, None]
    agents_by_seat[seat_a] = agent_a
    agents_by_seat[1 - seat_a] = agent_b

    def opponent_label(label: str) -> str | None:
        if label == agent_a.label:
            return agent_b.label
        if label == agent_b.label:
            return agent_a.label
        return None

    def opponent_seat(seat: int) -> int:
        return 1 - seat

    decks_by_seat = [None, None]
    decks_by_seat[seat_a] = agent_a.deck
    decks_by_seat[1 - seat_a] = agent_b.deck

    trace = JsonlTraceWriter(
        (trace_dir / f"match_{match_index:04d}.jsonl") if (debug and trace_dir is not None) else None
    )
    invalid_counts = Counter()
    agent_exception_counts = Counter()
    phase = "not_started"
    last_acting_label: str | None = None
    obs = None
    steps = 0
    reason = "unknown"
    prizes_initialized_seen = False

    try:
        phase = "battle_start"
        obs, start_data = battle_start(decks_by_seat[0], decks_by_seat[1])
        if obs is None:
            reason = f"battle_start failed: {to_plain(start_data, 3)!r}"
            return MatchResult(
                match_index=match_index,
                seat_a=seat_a,
                seat_b=1 - seat_a,
                winner_seat=None,
                winner_label=None,
                steps=0,
                turns=None,
                reason=reason,
                invalid_a=0,
                invalid_b=0,
                agent_exceptions_a=0,
                agent_exceptions_b=0,
            )

        if debug:
            trace.write({
                "event": "battle_start",
                "match": match_index,
                "seat_a": seat_a,
                "seat_b": 1 - seat_a,
                "start_data": to_plain(start_data, 4),
                "state": summarize_state(obs, name_table),
                "prizeLengths": prize_lengths(obs),
                "prizesInitializedSeen": prizes_initialized_seen,
                "select": summarize_select(obs, name_table),
            })

        for steps in range(1, max_steps + 1):
            if prizes_are_initialized(obs):
                prizes_initialized_seen = True

            finished, winner, finish_reason = infer_winner(
                obs, allow_prize_fallback=prizes_initialized_seen
            )
            if finished:
                reason = finish_reason
                label = agents_by_seat[winner].label if winner in (0, 1) else None
                return MatchResult(
                    match_index=match_index,
                    seat_a=seat_a,
                    seat_b=1 - seat_a,
                    winner_seat=winner,
                    winner_label=label,
                    steps=steps - 1,
                    turns=get_path(obs, "current.turn", default=None),
                    reason=reason,
                    invalid_a=invalid_counts[agent_a.label],
                    invalid_b=invalid_counts[agent_b.label],
                    agent_exceptions_a=agent_exception_counts[agent_a.label],
                    agent_exceptions_b=agent_exception_counts[agent_b.label],
                )

            select = get_select(obs)
            if select is None:
                # This should not happen after battle_start(deck0, deck1), but support it.
                player_index = get_your_index(obs)
            else:
                player_index = get_your_index(obs)

            # If CABT reports a forced-choice prompt with no options, do not call
            # the agent.  This usually means CABT is in a just-terminal state
            # that did not expose a winner/prize signal. Try no-Pokémon inference
            # one more time, then return unknown instead of counting an agent
            # invalid selection.
            n_options, min_count, _max_count = get_select_bounds(obs)
            if n_options == 0 and min_count > 0:
                forced_finished, forced_winner, forced_reason = no_pokemon_terminal(
                    obs, allow_terminal_fallback=prizes_initialized_seen
                )
                if forced_finished:
                    label = agents_by_seat[forced_winner].label if forced_winner in (0, 1) else None
                    return MatchResult(
                        match_index=match_index,
                        seat_a=seat_a,
                        seat_b=1 - seat_a,
                        winner_seat=forced_winner,
                        winner_label=label,
                        steps=steps - 1,
                        turns=get_path(obs, "current.turn", default=None),
                        reason=forced_reason,
                        invalid_a=invalid_counts[agent_a.label],
                        invalid_b=invalid_counts[agent_b.label],
                        agent_exceptions_a=agent_exception_counts[agent_a.label],
                        agent_exceptions_b=agent_exception_counts[agent_b.label],
                    )
                return MatchResult(
                    match_index=match_index,
                    seat_a=seat_a,
                    seat_b=1 - seat_a,
                    winner_seat=None,
                    winner_label=None,
                    steps=steps - 1,
                    turns=get_path(obs, "current.turn", default=None),
                    reason=f"forced_prompt_has_no_options:minCount={min_count}",
                    invalid_a=invalid_counts[agent_a.label],
                    invalid_b=invalid_counts[agent_b.label],
                    agent_exceptions_a=agent_exception_counts[agent_a.label],
                    agent_exceptions_b=agent_exception_counts[agent_b.label],
                    exception_source="empty_forced_select",
                    exception_message=f"select has minCount={min_count} but zero options",
                    exception=None,
                )

            acting_agent = agents_by_seat[player_index]
            if acting_agent is None:
                raise RuntimeError(f"No agent loaded for player seat {player_index}")

            if debug:
                trace.write({
                    "event": "pre_action",
                    "match": match_index,
                    "step": steps,
                    "actingSeat": player_index,
                    "actingAgent": acting_agent.label,
                    "state": summarize_state(obs, name_table),
                    "prizeLengths": prize_lengths(obs),
                    "prizesInitializedSeen": prizes_initialized_seen,
                    "select": summarize_select(obs, name_table),
                    "logs": summarize_logs(obs, name_table),
                })

            try:
                phase = "agent_call"
                last_acting_label = acting_agent.label
                selection = call_agent(acting_agent, obs)
            except Exception as e:
                invalid_counts[acting_agent.label] += 1
                agent_exception_counts[acting_agent.label] += 1
                tb = traceback.format_exc()
                selection = legal_fallback_selection(obs, rng)
                if debug:
                    trace.write({
                        "event": "agent_exception",
                        "match": match_index,
                        "step": steps,
                        "actingSeat": player_index,
                        "actingAgent": acting_agent.label,
                        "exception": repr(e),
                        "traceback": tb,
                        "fallbackSelection": selection,
                        "onInvalidSelection": on_invalid_selection,
                    })
                if on_invalid_selection == "loss":
                    winner = opponent_seat(player_index)
                    return MatchResult(
                        match_index=match_index,
                        seat_a=seat_a,
                        seat_b=1 - seat_a,
                        winner_seat=winner,
                        winner_label=agents_by_seat[winner].label,
                        steps=steps,
                        turns=get_path(obs, "current.turn", default=None),
                        reason=f"agent_exception_loss:{acting_agent.label}",
                        invalid_a=invalid_counts[agent_a.label],
                        invalid_b=invalid_counts[agent_b.label],
                        agent_exceptions_a=agent_exception_counts[agent_a.label],
                        agent_exceptions_b=agent_exception_counts[agent_b.label],
                        exception_source=f"agent_call:{acting_agent.label}",
                        exception_message=repr(e),
                        exception=tb,
                    )
                if on_invalid_selection == "exception":
                    return MatchResult(
                        match_index=match_index,
                        seat_a=seat_a,
                        seat_b=1 - seat_a,
                        winner_seat=None,
                        winner_label=None,
                        steps=steps,
                        turns=get_path(obs, "current.turn", default=None),
                        reason=f"agent_exception:{acting_agent.label}",
                        invalid_a=invalid_counts[agent_a.label],
                        invalid_b=invalid_counts[agent_b.label],
                        agent_exceptions_a=agent_exception_counts[agent_a.label],
                        agent_exceptions_b=agent_exception_counts[agent_b.label],
                        exception_source=f"agent_call:{acting_agent.label}",
                        exception_message=repr(e),
                        exception=tb,
                    )

            phase = "validate_selection"
            ok, why = validate_selection(selection, obs)
            if not ok:
                invalid_counts[acting_agent.label] += 1
                fallback = legal_fallback_selection(obs, rng)
                if debug:
                    trace.write({
                        "event": "invalid_selection",
                        "match": match_index,
                        "step": steps,
                        "actingSeat": player_index,
                        "actingAgent": acting_agent.label,
                        "selection": selection,
                        "reason": why,
                        "fallbackSelection": fallback,
                        "onInvalidSelection": on_invalid_selection,
                        "select": summarize_select(obs, name_table),
                        "logs": summarize_logs(obs, name_table),
                    })
                if on_invalid_selection == "loss":
                    winner = opponent_seat(player_index)
                    return MatchResult(
                        match_index=match_index,
                        seat_a=seat_a,
                        seat_b=1 - seat_a,
                        winner_seat=winner,
                        winner_label=agents_by_seat[winner].label,
                        steps=steps,
                        turns=get_path(obs, "current.turn", default=None),
                        reason=f"invalid_selection_loss:{acting_agent.label}:{why}",
                        invalid_a=invalid_counts[agent_a.label],
                        invalid_b=invalid_counts[agent_b.label],
                        agent_exceptions_a=agent_exception_counts[agent_a.label],
                        agent_exceptions_b=agent_exception_counts[agent_b.label],
                        exception_source="validate_selection",
                        exception_message=why,
                        exception=None,
                    )
                if on_invalid_selection == "exception":
                    return MatchResult(
                        match_index=match_index,
                        seat_a=seat_a,
                        seat_b=1 - seat_a,
                        winner_seat=None,
                        winner_label=None,
                        steps=steps,
                        turns=get_path(obs, "current.turn", default=None),
                        reason=f"invalid_selection:{acting_agent.label}:{why}",
                        invalid_a=invalid_counts[agent_a.label],
                        invalid_b=invalid_counts[agent_b.label],
                        agent_exceptions_a=agent_exception_counts[agent_a.label],
                        agent_exceptions_b=agent_exception_counts[agent_b.label],
                        exception_source="validate_selection",
                        exception_message=why,
                        exception=None,
                    )
                selection = fallback

            if debug:
                trace.write({
                    "event": "action",
                    "match": match_index,
                    "step": steps,
                    "actingSeat": player_index,
                    "actingAgent": acting_agent.label,
                    "selection": selection,
                })

            phase = "battle_select"
            try:
                obs = battle_select(selection)
            except IndexError as e:
                # CABT can reject a selection that passes simple index checks when
                # the option combination is semantically invalid. Attribute this to
                # the acting agent/fallback selection rather than to match logic.
                invalid_counts[acting_agent.label] += 1
                tb = traceback.format_exc()
                if debug:
                    trace.write({
                        "event": "battle_select_index_error",
                        "match": match_index,
                        "step": steps,
                        "actingSeat": player_index,
                        "actingAgent": acting_agent.label,
                        "selection": selection,
                        "select": summarize_select(obs, name_table),
                        "state": summarize_state(obs, name_table),
                        "exception": repr(e),
                        "traceback": tb,
                        "onInvalidSelection": on_invalid_selection,
                    })
                if on_invalid_selection == "loss":
                    winner = opponent_seat(player_index)
                    return MatchResult(
                        match_index=match_index,
                        seat_a=seat_a,
                        seat_b=1 - seat_a,
                        winner_seat=winner,
                        winner_label=agents_by_seat[winner].label,
                        steps=steps,
                        turns=get_path(obs, "current.turn", default=None),
                        reason=f"battle_select_index_error_loss:{acting_agent.label}",
                        invalid_a=invalid_counts[agent_a.label],
                        invalid_b=invalid_counts[agent_b.label],
                        agent_exceptions_a=agent_exception_counts[agent_a.label],
                        agent_exceptions_b=agent_exception_counts[agent_b.label],
                        exception_source="battle_select_index_error",
                        exception_message=repr(e),
                        exception=tb,
                    )
                raise

        # Max-step cutoff.
        reason = f"max_steps={max_steps} reached"
        return MatchResult(
            match_index=match_index,
            seat_a=seat_a,
            seat_b=1 - seat_a,
            winner_seat=None,
            winner_label=None,
            steps=max_steps,
            turns=get_path(obs, "current.turn", default=None),
            reason=reason,
            invalid_a=invalid_counts[agent_a.label],
            invalid_b=invalid_counts[agent_b.label],
            agent_exceptions_a=agent_exception_counts[agent_a.label],
            agent_exceptions_b=agent_exception_counts[agent_b.label],
        )

    except Exception as e:
        tb = traceback.format_exc()
        if debug:
            trace.write({"event": "match_exception", "match": match_index, "exception": repr(e), "traceback": tb})
        return MatchResult(
            match_index=match_index,
            seat_a=seat_a,
            seat_b=1 - seat_a,
            winner_seat=None,
            winner_label=None,
            steps=steps,
            turns=get_path(obs, "current.turn", default=None) if obs is not None else None,
            reason="exception",
            invalid_a=invalid_counts[agent_a.label],
            invalid_b=invalid_counts[agent_b.label],
            agent_exceptions_a=agent_exception_counts[agent_a.label],
            agent_exceptions_b=agent_exception_counts[agent_b.label],
            exception_source=phase if phase != "agent_call" else f"agent_call:{last_acting_label}",
            exception_message=repr(e),
            exception=tb,
        )
    finally:
        if save_visualize and trace_dir is not None:
            try:
                trace_dir.mkdir(parents=True, exist_ok=True)
                (trace_dir / f"match_{match_index:04d}_final_visualize.txt").write_text(
                    visualize_data(), encoding="utf-8"
                )
            except Exception:
                pass
        try:
            battle_finish()
        except Exception:
            pass
        trace.close()


# ---------------------------------------------------------------------------
# Reporting
# ---------------------------------------------------------------------------


def summarize_results(results: list[MatchResult], label_a: str, label_b: str) -> dict[str, Any]:
    wins = Counter(r.winner_label for r in results)
    total = len(results)
    completed = sum(1 for r in results if r.winner_label is not None)
    draws = total - completed
    a_wins = wins[label_a]
    b_wins = wins[label_b]
    exceptions = sum(1 for r in results if r.exception is not None)
    exception_sources = Counter(r.exception_source for r in results if r.exception_source)
    return {
        "matches": total,
        "completed": completed,
        "draws_or_unknown": draws,
        "exceptions": exceptions,
        "exception_sources": dict(exception_sources),
        "wins": {label_a: a_wins, label_b: b_wins},
        "win_rate_all": {
            label_a: a_wins / total if total else 0.0,
            label_b: b_wins / total if total else 0.0,
        },
        "win_rate_completed_only": {
            label_a: a_wins / completed if completed else 0.0,
            label_b: b_wins / completed if completed else 0.0,
        },
        "avg_steps": sum(r.steps for r in results) / total if total else 0.0,
        "invalid_selections": {
            label_a: sum(r.invalid_a for r in results),
            label_b: sum(r.invalid_b for r in results),
        },
        "agent_exceptions": {
            label_a: sum(r.agent_exceptions_a for r in results),
            label_b: sum(r.agent_exceptions_b for r in results),
        },
        "finish_reasons": dict(Counter(r.reason for r in results)),
    }


def write_results_csv(results: list[MatchResult], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(asdict(results[0]).keys()) if results else [])
        if results:
            writer.writeheader()
            for r in results:
                writer.writerow(asdict(r))


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------


def parse_card_ids(s: str | None) -> list[int]:
    if not s:
        return []
    out = []
    for part in re.split(r"[,\s]+", s.strip()):
        if part:
            out.append(int(part))
    return out


def main() -> None:
    p = argparse.ArgumentParser(description="Run CABT battles between two Python agent files.")
    p.add_argument("--agent-a", required=True, help="Path to first agent .py file")
    p.add_argument("--agent-b", required=True, help="Path to second agent .py file")
    p.add_argument("--deck-a", default=None, help="Optional path to agent A deck.csv")
    p.add_argument("--deck-b", default=None, help="Optional path to agent B deck.csv")
    p.add_argument("--label-a", default="A", help="Display label for agent A")
    p.add_argument("--label-b", default="B", help="Display label for agent B")
    p.add_argument("--matches", type=int, default=100, help="Number of matches to run")
    p.add_argument("--max-steps", type=int, default=20000, help="Safety cap per match")
    p.add_argument("--seed", type=int, default=12345, help="Python RNG seed for fallbacks/seating")
    p.add_argument(
        "--alternate-seats",
        action="store_true",
        help="Alternate which agent is player 0 each match. Strongly recommended.",
    )
    p.add_argument(
        "--seat-a",
        type=int,
        choices=[0, 1],
        default=0,
        help="Seat for agent A when not alternating seats.",
    )
    p.add_argument("--debug", action="store_true", help="Write detailed per-step JSONL traces")
    p.add_argument("--trace-dir", default="cabt_traces", help="Directory for debug traces/results")
    p.add_argument(
        "--save-visualize",
        action="store_true",
        help="Save visualize_data() text after each match; useful but can be slow/noisy.",
    )
    p.add_argument(
        "--dump-card-ids",
        default="1,2,3",
        help="Comma/space-separated card IDs to dump from all_card_data() when --debug is set.",
    )
    p.add_argument(
        "--no-metadata-dump",
        action="store_true",
        help="Disable all_attack/all_card_data metadata dumps even in debug mode.",
    )
    p.add_argument(
        "--abort-on-exception",
        action="store_true",
        help="Stop after the first match-level exception and print its source/message.",
    )
    p.add_argument(
        "--abort-on-invalid-selection",
        action="store_true",
        help="Stop after the first invalid agent selection, even when --on-invalid-selection=loss.",
    )
    p.add_argument(
        "--on-invalid-selection",
        choices=["loss", "fallback", "exception"],
        default="loss",
        help=(
            "How to handle invalid agent choices. 'loss' treats the acting agent as losing "
            "the match; 'fallback' uses a minimal legal-looking fallback and continues; "
            "'exception' records the match as failed. Default: loss."
        ),
    )
    p.add_argument(
        "--results-csv",
        default=None,
        help="Optional CSV path for per-match results. Defaults to trace-dir/results.csv.",
    )
    args = p.parse_args()

    rng = random.Random(args.seed)
    trace_dir = Path(args.trace_dir)
    if args.debug:
        trace_dir.mkdir(parents=True, exist_ok=True)

    # Set DEBUG_AGENT for imported agents too if user used --debug but did not export it.
    if args.debug:
        os.environ.setdefault("DEBUG_AGENT", "1")

    print(f"[load] importing agent A: {args.agent_a}")
    agent_a = load_agent(args.agent_a, args.label_a, args.deck_a)
    print(f"[load] importing agent B: {args.agent_b}")
    agent_b = load_agent(args.agent_b, args.label_b, args.deck_b)
    print(f"[deck] {agent_a.label}: {len(agent_a.deck)} cards from {agent_a.path}")
    print(f"[deck] {agent_b.label}: {len(agent_b.deck)} cards from {agent_b.path}")

    name_table = build_card_name_table()

    if args.debug and not args.no_metadata_dump:
        dump_card_metadata(parse_card_ids(args.dump_card_ids), trace_dir, stdout=True)
        dump_attack_metadata(trace_dir, stdout=True)

    results: list[MatchResult] = []
    start = time.time()
    for m in range(args.matches):
        if args.alternate_seats:
            seat_a = m % 2
        else:
            seat_a = args.seat_a
        r = run_match(
            match_index=m,
            agent_a=agent_a,
            agent_b=agent_b,
            seat_a=seat_a,
            debug=args.debug,
            trace_dir=trace_dir,
            max_steps=args.max_steps,
            rng=rng,
            name_table=name_table,
            save_visualize=args.save_visualize,
            on_invalid_selection=args.on_invalid_selection,
        )
        results.append(r)
        if args.abort_on_exception and r.exception is not None:
            print(f"[abort] match={m} exception_source={r.exception_source} message={r.exception_message}")
            print(r.exception or "")
            break
        if args.abort_on_invalid_selection and r.exception_source == "validate_selection":
            print(f"[abort-invalid] match={m} reason={r.reason} message={r.exception_message}")
            print("Inspect the match JSONL trace for event='invalid_selection'.")
            break
        if args.matches <= 20 or (m + 1) % max(1, args.matches // 10) == 0:
            print(
                f"[match {m+1}/{args.matches}] winner={r.winner_label} "
                f"steps={r.steps} reason={r.reason} invalid=({r.invalid_a},{r.invalid_b})"
            )

    elapsed = time.time() - start
    summary = summarize_results(results, agent_a.label, agent_b.label)
    summary["elapsed_sec"] = elapsed
    summary["matches_per_sec"] = len(results) / elapsed if elapsed > 0 else None

    results_csv = Path(args.results_csv) if args.results_csv else trace_dir / "results.csv"
    if results:
        write_results_csv(results, results_csv)
    summary_path = trace_dir / "summary.json"
    trace_dir.mkdir(parents=True, exist_ok=True)
    summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

    print("\n=== CABT battle summary ===")
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    print(f"[write] per-match results: {results_csv}")
    print(f"[write] summary: {summary_path}")
    if args.debug:
        print(f"[write] debug traces: {trace_dir}")


if __name__ == "__main__":
    main()

Overwriting battle_simulator.py


# Deck 1:

In [41]:
%%writefile deck1.py
import os
import sys
from collections import defaultdict
import glob

sys.path.append(glob.glob('/kaggle/input/**/cg-lib', recursive=True)[0])

from cg.api import AreaType, CardType, Log, LogType, Observation, SelectContext, OptionType, Card, Pokemon, State, all_card_data, to_observation_class

"""
Dragapult ex Deck
Advanced Level
This deck focuses on setting up multiple knockouts to take at least three Prize cards in a single turn with its Phantom Dive attack.
"""

# Load deck.csv in the dataset
file_path = "deck.csv"
if not os.path.exists(file_path):
    file_path = "/kaggle_simulations/agent/" + file_path
with open(file_path, "r") as file:
    csv = file.read().split("\n")
my_deck = []
for i in range(60):
    my_deck.append(int(csv[i]))
    
# Load all card data from the API's helper function
all_card = all_card_data()
# Create a lookup table (dictionary) to quickly access card data by its cardId
card_table = {c.cardId:c for c in all_card}

# Decklist
Dreepy = 119  # ×4
Drakloak = 120  # ×4
Dragapult_ex = 121  # ×3
Fezandipiti_ex = 140  # ×1
Latias_ex = 184  # ×1
Budew = 235  # ×2
Meowth_ex = 1071  # ×1
Rare_Candy = 1079  # ×2
Unfair_Stamp = 1080  # ×1
Buddy_Buddy_Poffin = 1086  # ×4
Night_Stretcher = 1097  # ×2
Crushing_Hammer = 1120  # ×4
Ultra_Ball = 1121  # ×4
Poke_Pad = 1152  # x3
Lucky_Helmet = 1156  # ×1
Boss_Orders = 1182  # ×3
Crispin = 1198  # ×4
Brock_Scouting = 1210  # ×2
Lillie_Determination = 1227  # ×4
Team_Rocket_Watchtower = 1256  # ×2
Basic_Fire_Energy = 2  # ×4
Basic_Psychic_Energy = 5  # ×4

UNNECESSARY = -10000000

class AttackPlan:
    attack: int = 0
    counter: list[int] = []

can_switch = False
can_attack = False
can_main_attack = False
can_energy_attach = False
use_support = 0  # The Supporter card planned for use.
bench_attacker = False  # Whether there is a Benched Pokémon that is ready to attack
pre_turn_log: list[Log] = []
current_turn_log: list[Log] = []

prize: list[int] = []
card_counts: defaultdict[int, int] = defaultdict(int)
serial_set: set[int] = set()
plan_a = AttackPlan()
plan_b = AttackPlan()


def no_damage_dex(id: int) -> bool:
    """Checks if the defending Pokémon possesses innate immunities preventing Dragapult ex from hitting it."""
    # Drednaw, Milotic ex, Sylveon, Crustle
    return id == 158 or id == 207 or id == 330 or id == 345


def no_damage_counter(pokemon: Pokemon) -> bool:
    """Checks if a target prevents placement of Phantom Dive's 6 bench damage counters (via abilities/Energy)."""
    # Poltchageist, Empoleon ex, Skeledirge, Milotic ex, Misty's Magikarp, Antique Cover Fossil
    if pokemon.id == 28 or pokemon.id == 199 or pokemon.id == 203 or pokemon.id == 207 or pokemon.id == 362 or pokemon.id == 1136:
        return True
    for card in pokemon.energyCards:
        # Mist Energy, Rock Fighting Energy
        if card.id == 11 or card.id == 20:
            return True
    return False


def prize_count(pokemon: Pokemon, is_attack_damage: bool) -> int:
    """Calculates how many Prize cards a Pokémon yields upon being Knocked Out, factoring in modifiers."""
    data = card_table[pokemon.id]
    count = 3 if data.megaEx else 2 if data.ex else 1
    if is_attack_damage:
        for card in pokemon.energyCards:
            if card.id == 12:  # Legacy Energy
                count -= 1
        for card in pokemon.tools:
            if card.id == 1172 and "Lillie" in data.name:  # Lillie’s Pearl
                count -= 1
    return max(0, count)


def pokemon_score(pokemon: Pokemon, is_attack_damage: bool) -> int:
    """Heuristically evaluates the tactical worth of targeting a specific Pokémon on the opponent's field."""
    data = card_table[pokemon.id]
    score = prize_count(pokemon, is_attack_damage) * 1000
    score += len(pokemon.energies) * 150
    score += len(pokemon.tools) * 100
    if data.stage2:
        score += 250
    elif data.stage1:
        score += 130
    
    id = pokemon.id
    # Squawkabilly ex, Noctowl, Fan Rotom, Archaludon ex
    if id == 144 or id == 322 or id == 323 or id == 337:
        score -= 200
    if id == 112 and len(pokemon.energies) >= 1:  # Munkidori
        score += 300
    score += pokemon.hp
    return score


def add_card_count(card: Card | Pokemon | None, my_index: int):
    if card == None:
        return
    if isinstance(card, Pokemon) or card.playerIndex == my_index:
        if card.serial not in serial_set:
            card_counts[card.id] -= 1
            serial_set.add(card.serial)
    if isinstance(card, Pokemon):
        for c in card.energyCards:
            add_card_count(c, my_index)
        for c in card.tools:
            add_card_count(c, my_index)
        for c in card.preEvolution:
            add_card_count(c, my_index)

def set_card_counts(obs: Observation, my_index: int):
    card_counts.clear()
    serial_set.clear()
    for id in my_deck:
        card_counts[id] += 1
    
    state = obs.current
    my_state = state.players[my_index]
    for card in my_state.hand:
        add_card_count(card, my_index)
    for card in my_state.discard:
        add_card_count(card, my_index)
    for card in my_state.bench:
        add_card_count(card, my_index)
    for card in my_state.active:
        add_card_count(card, my_index)
    for card in state.stadium:
        add_card_count(card, my_index)
    if state.looking != None:
        for card in state.looking:
            add_card_count(card, my_index)
    add_card_count(obs.select.effect, my_index)

    
def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    """Helper function to safely extract a Card or Pokemon object from specific zones."""
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK:
            return obs.select.deck[index]
        case AreaType.HAND:
            return ps.hand[index]
        case AreaType.DISCARD:
            return ps.discard[index]
        case AreaType.ACTIVE:
            return ps.active[index]
        case AreaType.BENCH:
            return ps.bench[index]
        case AreaType.PRIZE:
            return ps.prize[index]
        case AreaType.STADIUM:
            return obs.current.stadium[index]
        case AreaType.LOOKING:
            return obs.current.looking[index]
        case _:
            return None

def main_option_proc(obs: Observation, damage: int):
    state = obs.current
    select = obs.select
    my_index = state.yourIndex
    my_state = state.players[my_index]
    op_state = state.players[1 - my_index]

    global can_switch
    global can_attack
    global can_main_attack
    global can_energy_attach

    can_switch = False
    can_attack = False
    can_main_attack = False
    can_energy_attach = False
    for o in select.option:
        if o.type == OptionType.RETREAT:
            can_switch = True
        elif o.type == OptionType.ATTACK:
            can_attack = True
            if o.attackId == 154:  # Phantom Dive
                can_main_attack = True
    
    plan_a.attack = -1
    plan_b.attack = -1
    if not can_main_attack and not (bench_attacker and can_switch):
        return
    
    cards = [op_state.active[0]]
    for pokemon in op_state.bench:
        cards.append(pokemon)
    counter_indices = []
    ci = []
    ci.append(0)
    remain_damage = 60
    while ci:
        index = ci[-1]
        hp = cards[index].hp
        if remain_damage >= hp:
            counter_indices.append(ci.copy())
            if index < len(cards) - 1:
                remain_damage -= hp
                ci.append(index + 1)
                continue
        if index == len(cards) - 1:
            ci.pop()
            if ci:
                remain_damage += cards[ci[-1]].hp
        if ci:
            ci[-1] += 1
    counter_indices.append([])

    remain_prize = len(my_state.prize)
    plan_score = 0
    for i, pokemon in enumerate(cards):
        base_prize_count = 0
        base_score = pokemon_score(pokemon, True)
        active_damage = 0 if no_damage_dex(pokemon.id) else damage
        if pokemon.hp <= active_damage:
            base_prize_count += prize_count(pokemon, True)
        else:
            base_score *= active_damage / pokemon.hp
        ci = []
        max_score = base_score
        if remain_prize <= base_prize_count:
            max_score = 50000
        else:
            for indices in counter_indices:
                if i in indices:
                    continue
                prize = base_prize_count
                score = base_score
                for index in indices:
                    prize += prize_count(cards[index], False)
                    score += pokemon_score(cards[index], False)
                if remain_prize <= prize:
                    score = 50000
                else:
                    if prize >= 2:
                        if remain_prize <= 4:
                            score -= 1200
                    elif prize == 1:
                        score -= 300
                    else:
                        score += 1200
                if max_score < score:
                    max_score = score
                    ci = indices
        if plan_score < max_score:
            plan_score = max_score
            plan_a.attack = i
            plan_a.counter = ci
        if i == 0:
            plan_b.attack = plan_a.attack
            plan_b.counter = plan_a.counter

def agent(obs_dict: dict) -> list[int]:
    """Main Agent Function.

    Each element in the returned list must be >= 0 and < len(obs.select.option).
    The list length must be between obs.select.minCount and obs.select.maxCount (inclusive), with no duplicate elements.
    
    Returns:
        list[int]: A list of option index.
    """
    obs = to_observation_class(obs_dict)
    if obs.select == None:
        # In the initial selection, the obs.select is None, and it is necessary to return the deck.
        # The deck is a list of 60 card IDs.
        # The deck must comply with the Pokémon Trading Card Game rules.
        return my_deck

    global pre_turn_log
    global current_turn_log

    state = obs.current
    select = obs.select
    context = select.context
    my_index = state.yourIndex
    my_state = state.players[my_index]
    op_state = state.players[1 - my_index]
            
    if state.turn == 0:
        prize.clear()
        pre_turn_log.clear()
        current_turn_log.clear()
    else:
        for log in obs.logs:
            current_turn_log.append(log)
            if log.type == LogType.TURN_END:
                pre_turn_log = current_turn_log
                current_turn_log = []

    pre_ko = False
    no_item = False
    for log in pre_turn_log:
        if log.type == LogType.ATTACK:
            if log.attackId == 323:  # Itchy Pollen
                no_item = True
        elif log.type == LogType.MOVE_CARD:
            if (log.playerIndex == my_index
                and (log.fromArea == AreaType.BENCH or log.fromArea == AreaType.ACTIVE)
                and log.toArea == AreaType.DISCARD):
                pre_ko = True

    if select.deck != None:
        set_card_counts(obs, my_index)
        for card in select.deck:
            card_counts[card.id] -= 1
        prize.clear()
        for id in card_counts:
            for _ in range(card_counts[id]):
                prize.append(id)
                
    set_card_counts(obs, my_index)
    for id in prize:
        card_counts[id] -= 1
    deck_counts = card_counts

    prize_diff = len(my_state.prize) - len(op_state.prize)
    
    global bench_attacker

    # Number of cards per card ID on the Bench and in the Active Spot
    field_counts = defaultdict(int)
    # Number of cards per card ID in hand
    hand_counts = defaultdict(int)
    # Number of cards per card ID in discard pile
    discard_counts = defaultdict(int)
    
    active_id = 0
    bench_attacker = False
    can_evolve_dreepy = False
    evolve_dreepy_count = 0
    can_evolve_drakloak = False
    damage = 200
    for card in my_state.active:
        if card == None:
            continue
        active_id = card.id
        field_counts[card.id] += 1
        if not card.appearThisTurn:
            if card.id == Dreepy:
                can_evolve_dreepy = True
                evolve_dreepy_count += 1
            elif card.id == Drakloak:
                can_evolve_drakloak = True
    for card in my_state.bench:
        field_counts[card.id] += 1
        if not card.appearThisTurn:
            if card.id == Dreepy:
                can_evolve_dreepy = True
                evolve_dreepy_count += 1
            elif card.id == Drakloak:
                can_evolve_drakloak = True
        if card.id == Dragapult_ex and len(card.energies) >= 2:
            bench_attacker = True
    main_pokemon_count = field_counts[Dreepy] + field_counts[Drakloak] + field_counts[Dragapult_ex]
    no_more_dex = (field_counts[Dragapult_ex] * 2 >= len(op_state.prize))

    stadium_id = 0
    for card in state.stadium:
        stadium_id = card.id

    support_count = 0

    for card in my_state.discard:
        discard_counts[card.id] += 1

    def attach_score(attach_id: int, pokemon: Pokemon, active: bool) -> int:
        energy_count = len(pokemon.energies)
        if card_table[attach_id].cardType == CardType.TOOL:
            # Attach tool
            score = 60000
            if active:
                score += 1000
            return score
        
        # Attach energy
        if pokemon.id == Budew:
            return -1
        elif pokemon.id == Meowth_ex or pokemon.id == Fezandipiti_ex or pokemon.id == Latias_ex:
            if active and not can_switch and not my_state.asleep and not my_state.paralyzed:
                if bench_attacker or field_counts[Budew] >= 1:
                    return 22000
                else:
                    return 18000
            else:
                return -1
        if active and can_main_attack:
            return -1
        score = 20000
        if energy_count >= 2:
            if active and not can_switch and not my_state.asleep and not my_state.paralyzed:
                score += 200
            else:
                return -1
        elif energy_count == 1:
            if attach_id == pokemon.energyCards[0].id:
                return -1
            if pokemon.id == Dragapult_ex:
                score += 250
            elif pokemon.id == Dreepy:
                score -= 150
            else:
                score -= 200
            if active:
                score += 200
        else:  # energy_count == 0
            if active:
                if bench_attacker:
                    score += 400
            else:
                if pokemon.id == Dragapult_ex:
                    score += 150
                elif pokemon.id == Dreepy:
                    score += 100
                else:
                    score += 50
                if bench_attacker:
                    score -= 200
        if no_more_dex and (pokemon.id == Dreepy or pokemon.id == Drakloak):
            score -= 500
        return score
    
    def hand_score(id: int, ignore_count: bool):
        score = 0
        if id == Dreepy:
            if main_pokemon_count >= 3:
                score = 1000
            else:
                score = 18000
        elif id == Drakloak:
            if can_evolve_dreepy:
                score = 20000
            else:
                score = 3000
        elif id == Dragapult_ex:
            if no_more_dex:
                score = UNNECESSARY
            elif can_evolve_dreepy and hand_counts[Rare_Candy] >= 1 and not no_item:
                score = 40000
            elif can_evolve_drakloak:
                if field_counts[id] == 0:
                    score = 30000
                elif field_counts[id] == 1:
                    score = 10000
                else:
                    score = 50
            else:
                if field_counts[id] >= 2:
                    score = 50
                else:
                    score = 2000
        elif id == Fezandipiti_ex:
            if pre_ko:
                score = 50000
            elif prize_diff <= -2:
                score = 5
            elif len(op_state.prize) == 1:
                score = UNNECESSARY
        elif id == Latias_ex:
            if active_id == Fezandipiti_ex or active_id == Meowth_ex or active_id == Dreepy:
                if field_counts[Drakloak] + field_counts[Dragapult_ex] == 0:
                    score = 28000
                else:
                    score = 15000
            else:
                score = 10
        elif id == Budew:
            if field_counts[id] + field_counts[Drakloak] + field_counts[Dragapult_ex] >= 1:
                score = UNNECESSARY
            elif state.turn >= 2:
                score = 30000
        elif id == Meowth_ex:
            if support_count > hand_counts[Boss_Orders] or stadium_id == Team_Rocket_Watchtower:
                score = 5
            elif state.supporterPlayed:
                score = 40
            else:
                score = 35000
        elif id == Rare_Candy:
            if no_more_dex:
                score = UNNECESSARY
            elif can_evolve_dreepy and hand_counts[Dragapult_ex] >= 1:
                score = 40000
        elif id == Unfair_Stamp:
            if pre_ko:
                score = 80000
            elif len(op_state.prize) == 1:
                score = UNNECESSARY
            else:
                score = 80
        elif id == Buddy_Buddy_Poffin:
            count = deck_counts[Dreepy]
            if count == 0:
                score = UNNECESSARY
            else:
                if state.turn <= 2 and field_counts[Budew] == 0 and deck_counts[Budew] >= 1:
                    count += 1
                if count >= 2:
                    score = 35000
        elif id == Night_Stretcher:
            for i in discard_counts:
                if discard_counts[i] >= 1:
                    card_type = card_table[i].cardType
                    if card_type == CardType.POKEMON or card_type == CardType.BASIC_ENERGY:
                        score = max(score, hand_score(i, ignore_count))
        elif id == Crushing_Hammer:
            score = 20
        elif id == Ultra_Ball:
            if main_pokemon_count <= 2 or field_counts[Dreepy] >= 1:
                score = 70
            else:
                score = 5
        elif id == Poke_Pad:
            score = max(hand_score(Dreepy, ignore_count), hand_score(Drakloak, ignore_count))
        elif id == Lucky_Helmet:
            score = 15
        elif id == Boss_Orders:
            if plan_a.attack > 0:
                score = 60000
        elif id == Crispin:
            if not ignore_count or support_count == 0:
                if deck_counts[Basic_Fire_Energy] == 0 or deck_counts[Basic_Psychic_Energy] == 0:
                    score = 10
                if not can_main_attack and not bench_attacker and field_counts[Dragapult_ex] >= 1:
                    score = 55000
                else:
                    score = 25000
        elif id == Brock_Scouting:
            if not ignore_count or support_count == 0:
                if state.turn == 2 and field_counts[Budew] + field_counts[Latias_ex] == 0:
                    score = 50000
                else:
                    score = 30000
        elif id == Lillie_Determination:
            if not ignore_count or support_count == 0:
                score = 45000
        elif id == Team_Rocket_Watchtower:
            if stadium_id != 0 and stadium_id != Team_Rocket_Watchtower:
                score = 4000
        elif id == Basic_Fire_Energy or id == Basic_Psychic_Energy:
            if can_main_attack and (len(op_state.prize) <= 2
                or (bench_attacker and len(op_state.prize) <= 4)):
                score = UNNECESSARY
            else:
                max_score = -10000
                for pokemon in my_state.active:
                    if pokemon == None:
                        continue
                    max_score = max(max_score, attach_score(id, pokemon, True))
                for pokemon in my_state.bench:
                    max_score = max(max_score, attach_score(id, pokemon, False))
                score = max_score - 5000
                if can_main_attack or bench_attacker:
                    score /= 10
        
        if not ignore_count and hand_counts[id] > 0:
            if id == Drakloak and hand_counts[id] < evolve_dreepy_count:
                score -= 10
            elif id == Dreepy:
                score -= 100
            else:
                score -= 100000
        return score

    global use_support
    if context == SelectContext.MAIN:
        main_option_proc(obs, damage)
                    
        use_support = 0
        if not state.supporterPlayed:
            support_score = 0
            for o in select.option:
                if o.type == OptionType.PLAY:
                    card = get_card(obs, AreaType.HAND, o.index, state.yourIndex)
                    if card_table[card.id].cardType == CardType.SUPPORTER:
                        score = hand_score(card.id, True)
                        if support_score < score:
                            support_score = score
                            use_support = card.id

    hand_scores = []
    negative_hand_count = 0
    for card in my_state.hand:
        score = hand_score(card.id, False)
        hand_scores.append(score)
        if score < 0:
            negative_hand_count += 1
        hand_counts[card.id] += 1
        if card_table[card.id].cardType == CardType.SUPPORTER and card.id != Boss_Orders:
            support_count += 1

    no_draw = (my_state.deckCount <= 8)  # Whether to restrict actions that reduce the deck
    do_switch = (not can_main_attack and (bench_attacker or (active_id != Budew and field_counts[Budew] >= 1 and state.turn >= 2)))
    effect_card_id = 0 if select.effect == None else select.effect.id
    context_card_id = 0 if select.contextCard == None else select.contextCard.id
    
    scores = []  # Score for each action
    for o in select.option:
        score = 0  # The default and baseline score is 0.
        if o.type == OptionType.NUMBER:
            score = o.number
        elif o.type == OptionType.YES:
            if context == SelectContext.IS_FIRST:
                score = -1
            else:
                score = 1
        elif o.type == OptionType.CARD:
            card = get_card(obs, o.area, o.index, o.playerIndex)
            if card != None:
                energy_count = 0
                hp = 0
                if isinstance(card, Pokemon):
                    energy_count = len(card.energies)
                    hp = card.hp
                if (context == SelectContext.SWITCH
                    or context == SelectContext.TO_ACTIVE
                    or context == SelectContext.SETUP_ACTIVE_POKEMON):
                    # Selection of the Pokémon to send to the Active Spot
                    if o.playerIndex == my_index:
                        if card.id == Dreepy:
                            score += 10000
                        elif card.id == Drakloak:
                            if energy_count >= 1:
                                score += 20000
                            else:
                                score -= 10000
                        elif card.id == Dragapult_ex:
                            score += 50000
                        elif card.id == Budew:
                            if context != SelectContext.SWITCH:
                                score += 100000
                            elif not bench_attacker:
                                score += 30000
                        elif card.id == Fezandipiti_ex:
                            score -= 1000
                        elif card.id == Meowth_ex:
                            score -= 2000
                    else:
                        if plan_a.attack == o.index + 1:
                            score += 100000
                    score += energy_count * 1000
                    score += hp
                elif context == SelectContext.SETUP_BENCH_POKEMON:
                    if my_index == state.firstPlayer or card.id != Dreepy:
                        score = -1
                elif context == SelectContext.TO_BENCH or context == SelectContext.TO_HAND:
                    score = hand_score(card.id, False)
                    hand_counts[card.id] += 1
                    if effect_card_id == Crispin:
                        # Reverse scoring
                        score = 100000 - hand_score(card.id, True)
                elif context == SelectContext.DISCARD:
                    hand_counts[card.id] -= 1
                    if card_table[card.id].cardType == CardType.SUPPORTER:
                        support_count -= 1
                    score = -hand_score(card.id, False)
                elif context == SelectContext.DAMAGE_COUNTER or context == SelectContext.DAMAGE_COUNTER_ANY:
                    if hp > 0:
                        score = 100000 - 10 * hp + pokemon_score(card, False)
                        if context == SelectContext.DAMAGE_COUNTER:
                            if 210 <= hp <= 230:
                                score += 20000 + hp * 20
                                if o.area == AreaType.ACTIVE:
                                    score += 10000
                            elif 40 <= hp <= 90:
                                score += 10000 + hp * 20
                            elif hp <= 30:
                                score += -10000 + hp * 20
                            if card.id == 133 or card.id == 351:
                                score += 30000
                        else:
                            index = o.index + 1
                            if index in plan_b.counter:
                                score += 100000
                            else:
                                remain_damage = select.remainDamageCounter * 10
                                if 210 <= hp <= 200 + remain_damage:
                                    score += 30000
                                elif 20 <= hp <= 60 + remain_damage:
                                    score += 10000
                                elif hp == 10:
                                    score -= 100000
                            if no_damage_counter(card):
                                score = -1
                elif context == SelectContext.ATTACH_FROM:
                    score = attach_score(context_card_id, card, o.area == AreaType.ACTIVE)
                    if card.id == Dragapult_ex:
                        score += 200
        elif o.type == OptionType.ENERGY_CARD or o.type == OptionType.ENERGY:
            # Discarding energy (Retreat or Crushing Hammer)
            if o.playerIndex != state.yourIndex:
                if o.area == AreaType.BENCH:
                    score = 20
                else:
                    score = 10
                card = get_card(obs, o.area, o.index, o.playerIndex)
                if card_table[card.id].cardType == CardType.SPECIAL_ENERGY:
                    score += 1
        elif o.type == OptionType.PLAY:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            card_score = hand_scores[o.index]
            if card.id == Dreepy:
                score = 51000
            elif card.id == Fezandipiti_ex:
                if card_score > 0:
                    score = 53000
                else:
                    score = -1
            elif card.id == Latias_ex:
                if active_id != Drakloak and active_id != Dragapult_ex:
                    score = 51000
                else:
                    score = -1
            elif card.id == Budew:
                if field_counts[Budew] == 0 and field_counts[Dragapult_ex] == 0:
                    score = 52000
                else:
                    score = -1
            elif card.id == Meowth_ex:
                if state.supporterPlayed or stadium_id == Team_Rocket_Watchtower:
                    score = -1
                elif support_count == 0:
                    score = 50000
                elif support_count == hand_counts[Boss_Orders] and not plan_a.attack <= 0:
                    score = 50000
                else:
                    score = -1
            elif card.id == Rare_Candy:
                if no_more_dex:
                    score = -1
                else:
                    score = 75000
            elif card.id == Unfair_Stamp:
                score = 15000
            elif card.id == Night_Stretcher:
                if card_score >= 18000:
                    score = 42000
                else:
                    score = -1
            elif card.id == Crushing_Hammer:
                score = 40000
            elif card.id == Boss_Orders:
                if card.id == use_support:
                    score = 35000
                else:
                    score = -1
            elif card.id == Lillie_Determination:
                if card.id == use_support:
                    score = 14000
                else:
                    score = -1
            elif card.id == Team_Rocket_Watchtower:
                if stadium_id > 0 or state.turn == 1:
                    score = 80000
                else:
                    score = -1
            elif no_draw:
                score = -1
            elif card.id == Buddy_Buddy_Poffin:
                if deck_counts[Dreepy] > 0:
                    score = 46000
                else:
                    score = -1
            elif card.id == Ultra_Ball:
                if negative_hand_count >= 2:
                    score = 44000
                else:
                    score = -1
            elif card.id == Poke_Pad:
                if deck_counts[Dreepy] + deck_counts[Drakloak] > 0:
                    score = 45000
                else:
                    score = -1
            elif card.id == Crispin or card.id == Brock_Scouting:
                if card.id == use_support:
                    score = 35000
                else:
                    score = -1
        elif o.type == OptionType.ATTACH:
            card = get_card(obs, o.area, o.index, my_index)
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score = attach_score(card.id, pokemon, o.inPlayArea == AreaType.ACTIVE)
        elif o.type == OptionType.EVOLVE:
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score += len(pokemon.energies)
            if pokemon.id == Dreepy:
                score += 30000
            elif field_counts[Dragapult_ex] >= 2 or (field_counts[Dragapult_ex] == 1 and len(op_state.prize) <= 2):
                score = -1
            else:
                score += 70000
        elif o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if no_draw:
                score = -1
            elif card.id == 1267:  # Lumiose City
                score = 1
            else:
                score = 40000
        elif o.type == OptionType.RETREAT:
            if do_switch:
                score = 10000
            else:
                score = -1
        elif o.type == OptionType.ATTACK:
            score = o.attackId

        scores.append(score)

    output = []
    if len(scores) >= 1:
        # Select in descending order of score
        sorted_scores = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        for i in range(select.maxCount):
            # If the score is negative, do not select it if skipping is possible
            if (sorted_scores[i][1] >= 0
                or select.minCount > i
                or (context != SelectContext.TO_BENCH and context != SelectContext.SETUP_BENCH_POKEMON)):
                output.append(sorted_scores[i][0])
                
    return output

Overwriting deck1.py


# Deck 2:

In [42]:
%%writefile deck2.py
import os
import sys
from collections import defaultdict
import glob

sys.path.append(glob.glob('/kaggle/input/**/cg-lib', recursive=True)[0])

from cg.api import AreaType, CardType, EnergyType, Observation, SelectContext, OptionType, Card, Pokemon, all_card_data, to_observation_class

"""
Mega Lucario ex Deck
Intermediate Level
This deck battles by strategically switching between Mega Lucario ex as the main attacker, and Hariyama and Solrock as secondary attackers.
"""

# Load deck.csv in the dataset
file_path = "deck.csv"
if not os.path.exists(file_path):
    file_path = "/kaggle_simulations/agent/" + file_path
with open(file_path, "r") as file:
    csv = file.read().split("\n")
my_deck = []
for i in range(60):
    my_deck.append(int(csv[i]))

# Fetch card metadata database and create an ID-to-Card lookup table
all_card = all_card_data()
card_table = {c.cardId:c for c in all_card}

# Decklist
Makuhita = 673  # ×2
Hariyama = 674  # ×2
Lunatone = 675  # ×2
Solrock = 676  # ×3
Riolu = 677  # ×3
Mega_Lucario_ex = 678  # ×4
Dusk_Ball = 1102  # ×4
Switch = 1123  # ×2
Premium_Power_Pro = 1141  # ×4
Fighting_Gong = 1142  # ×4
Poke_Pad = 1152  # x4
Hero_Cape = 1159  # ×1
Boss_Orders = 1182  # ×2
Carmine = 1192  # ×4
Lillie_Determination = 1227  # ×4
Gravity_Mountain = 1252  # ×2
Basic_Fighting_Energy = 6  # ×13


class AttackPlan:
    attacker = -1
    target = -1
    attack_index = -1
    remain_hp = -1
    energy = False


plan = AttackPlan()
pre_turn = 0
ability_used = False


def get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:
    """Helper function to safely extract a Card or Pokemon object from specific zones."""
    ps = obs.current.players[player_index]
    match area:
        case AreaType.DECK:
            return obs.select.deck[index]
        case AreaType.HAND:
            return ps.hand[index]
        case AreaType.DISCARD:
            return ps.discard[index]
        case AreaType.ACTIVE:
            return ps.active[index]
        case AreaType.BENCH:
            return ps.bench[index]
        case AreaType.PRIZE:
            return ps.prize[index]
        case AreaType.STADIUM:
            return obs.current.stadium[index]
        case AreaType.LOOKING:
            return obs.current.looking[index]
        case _:
            return None


def prize_count(pokemon: Pokemon) -> int:
    """Calculates how many Prize cards a Pokémon yields upon being Knocked Out, factoring in modifiers."""
    data = card_table[pokemon.id]
    count = 3 if data.megaEx else 2 if data.ex else 1
    for card in pokemon.energyCards:
        if card.id == 12:  # Legacy Energy
            count -= 1
    for card in pokemon.tools:
        if card.id == 1172 and "Lillie" in data.name:  # Lillie’s Pearl
            count -= 1
    return max(0, count)


def pokemon_score(pokemon: Pokemon) -> int:
    """Heuristically evaluates the tactical worth of targeting a specific Pokémon on the opponent's field."""
    data = card_table[pokemon.id]
    score = prize_count(pokemon) * 1000
    score += len(pokemon.energies) * 150
    score += len(pokemon.tools) * 100
    if data.stage2:
        score += 250
    elif data.stage1:
        score += 130
    
    id = pokemon.id
    # Squawkabilly ex, Noctowl, Fan Rotom, Archaludon ex
    if id == 144 or id == 322 or id == 323 or id == 337:
        score -= 200
    if id == 112 and len(pokemon.energies) >= 1:  # Munkidori
        score += 300
    score += pokemon.hp
    return score


def agent(obs_dict: dict) -> list[int]:
    """Main Agent Function.

    Each element in the returned list must be >= 0 and < len(obs.select.option).
    The list length must be between obs.select.minCount and obs.select.maxCount (inclusive), with no duplicate elements.
    
    Returns:
        list[int]: A list of option index.
    """
    obs = to_observation_class(obs_dict)
    if obs.select == None:
        # In the initial selection, the obs.select is None, and it is necessary to return the deck.
        # The deck is a list of 60 card IDs.
        # The deck must comply with the Pokémon Trading Card Game rules.
        return my_deck
        
    state = obs.current
    select = obs.select
    context = select.context
    my_index = state.yourIndex
    my_state = state.players[my_index]
    op_state = state.players[1 - my_index]
    my_prize = len(my_state.prize)

    global plan
    global pre_turn
    global ability_used
    if pre_turn != state.turn:
        pre_turn = state.turn
        plan = AttackPlan()
        ability_used = False
            
    field_counts = defaultdict(int)  # Number of cards per card ID on the Bench and in the Active Spot
    hand_counts = defaultdict(int)  # Number of cards per card ID in hand
    discard_counts = defaultdict(int)  # Number of cards per card ID in discard pile

    attacker1 = False
    attacker2 = False
    for card in my_state.active + my_state.bench:
        if card == None:
            continue
        field_counts[card.id] += 1
        if card.id == Makuhita or card.id == Hariyama:
            if len(card.energies) >= 3:
                attacker2 = True
        elif card.id == Riolu or card.id == Mega_Lucario_ex:
            if len(card.energies) >= 2:
                attacker1 = True

    for card in my_state.hand:
        hand_counts[card.id] += 1

    for card in my_state.discard:
        discard_counts[card.id] += 1

    stadium_id = 0
    for card in state.stadium:
        stadium_id = card.id
            
    can_attack = False
    if context == SelectContext.MAIN:
        can_switch = False
        can_op_switch = False
        can_use_mega_brave = False
        for o in select.option:
            if o.type == OptionType.PLAY:
                card = get_card(obs, AreaType.HAND, o.index, my_index)
                if card.id == Switch:
                    can_switch = True
                elif card.id == Boss_Orders:
                    can_op_switch = True
            elif o.type == OptionType.EVOLVE:
                card = get_card(obs, AreaType.HAND, o.index, my_index)
                if card.id == Hariyama:
                    can_op_switch = True
            elif o.type == OptionType.RETREAT:
                can_switch = True
            elif o.type == OptionType.ATTACK:
                can_attack = True
                if o.attackId == 983:  # Mega Brave
                    can_use_mega_brave = True
        
        my_cards = [my_state.active[0]]
        for pokemon in my_state.bench:
            my_cards.append(pokemon)
        op_cards = [op_state.active[0]]
        for pokemon in op_state.bench:
            op_cards.append(pokemon)

        if state.turn >= 2:
            best_score = -1
            for i, my_pokemon in enumerate(my_cards):
                if i != 0 and not can_switch:
                    break
                for a in range(2):
                    energy_required = 0
                    base_damage = 0
                    base_score = 0
                    if my_pokemon.id == Mega_Lucario_ex:
                        if a == 0:
                            energy_required = 1
                            base_damage = 130
                            base_score += 60 * min(3, discard_counts[Basic_Fighting_Energy])
                        else:
                            energy_required = 2
                            base_damage = 270
                        if my_prize == 2 or my_prize == 3:
                            base_score -= 500
                    elif a == 1:
                        break
                    elif my_pokemon.id == Hariyama:
                        energy_required = 3
                        base_damage = 210
                    elif my_pokemon.id == Makuhita:
                        for o in select.option:
                            if o.type == OptionType.EVOLVE:
                                index = o.inPlayIndex
                                if o.inPlayArea == AreaType.BENCH:
                                    index += 1
                                if index == i:
                                    break
                        else:
                            break
                        base_score -= 100
                        energy_required = 3
                        base_damage = 210
                    elif my_pokemon.id == Solrock:
                        if field_counts[Lunatone] >= 1:
                            energy_required = 1
                            base_damage = 70
                    
                    if base_damage <= 0:
                        continue
                    
                    more_energy = False
                    energy_count = len(my_pokemon.energies)
                    if a == 1 and i == 0 and energy_count >= 2 and not can_use_mega_brave:
                        break
                    if energy_count < energy_required:
                        if hand_counts[Basic_Fighting_Energy] >= 1 and not state.energyAttached:
                            energy_count += 1
                            if energy_count < energy_required:
                                continue
                            else:
                                more_energy = True
                        else:
                            continue

                    for j, op_pokemon in enumerate(op_cards):
                        if j != 0 and not can_op_switch:
                            break
                        damage = base_damage
                        data = card_table[op_pokemon.id]
                        if data.weakness == EnergyType.FIGHTING:
                            damage *= 2
                        elif data.resistance == EnergyType.FIGHTING:
                            damage -= 30
                        prize = 0
                        score = pokemon_score(op_pokemon)
                        if op_pokemon.hp <= damage:
                            prize = prize_count(op_pokemon)
                        else:
                            score *= damage / op_pokemon.hp
                        score += base_score
                            
                        if len(op_state.prize) <= prize:
                            score = 50000
                        
                        if i == 0:
                            score += 220
                        if j == 0:
                            score += 300
                        score += energy_count
                        if best_score < score:
                            best_score = score
                            plan.attacker = i
                            plan.target = j
                            plan.attack_index = a
                            plan.remain_hp = op_pokemon.hp - damage
                            plan.energy = more_energy
    
    # Attach energy score
    def energy_score(pokemon: Pokemon, active: bool) -> int:
        energy_count = len(pokemon.energies)
        score = 8000
        if active:
            score += 10
        if pokemon.id == Makuhita or pokemon.id == Hariyama:
            if pokemon.id == Hariyama:
                score += 1
            if energy_count < 3:
                score += 100
            if attacker2:
                score -= 50
        elif pokemon.id == Lunatone:
            score -= 100
        elif pokemon.id == Solrock:
            if energy_count < 1:
                score += 20
            else:
                score -= 100
        elif pokemon.id == Riolu or pokemon.id == Mega_Lucario_ex:
            if pokemon.id == Mega_Lucario_ex:
                score += 1
            if energy_count < 2:
                score += 100
            if attacker1:
                score -= 50
        return score

    # Iterate over every possible option and assign a heuristic score.
    scores = []  # Score for each action
    for o in select.option:
        score = 0  # The default and baseline score is 0.
        if o.type == OptionType.NUMBER:
            score = o.number  # e.g., for "draw X cards"
        elif o.type == OptionType.YES:
            score = 1  # Prefer "Yes"
        elif o.type == OptionType.CARD:
            card = get_card(obs, o.area, o.index, o.playerIndex)
            if card != None:
                energy_count = 0
                if isinstance(card, Pokemon):
                    energy_count = len(card.energies)
                if context == SelectContext.SWITCH or context == SelectContext.TO_ACTIVE:
                    # Selection of the Pokémon to send to the Active Spot
                    if o.playerIndex == my_index:
                        score += energy_count * 2
                        if o.index == plan.attacker - 1:
                            score += 100
                        if card.id == Mega_Lucario_ex:
                            if my_prize == 2 or my_prize == 3:
                                score += 8
                            else:
                                score += 20
                        elif card.id == Hariyama and energy_count >= 2:
                            score += 15
                        elif card.id == Makuhita and energy_count >= 2:
                            score += 10
                        elif card.id == Solrock:
                            score += 5
                        elif card.id == Riolu:
                            score += 4
                    else:
                        if o.index == plan.target - 1:
                            score += 100
                elif context == SelectContext.SETUP_ACTIVE_POKEMON:
                    # Prioritize playing Riolu if going first, and Solrock if going second.
                    if card.id == Solrock:
                        if state.firstPlayer == my_index:
                            score = 2
                        else:
                            score = 4
                    elif card.id == Riolu:
                        score = 3
                    elif card.id == Makuhita:
                        score = 1
                elif context == SelectContext.TO_HAND:
                    score = 200 - hand_counts[card.id] * 100
                    if card.id == Makuhita:
                        if field_counts[card.id] >= 1:
                            score -= 10
                        else:
                            score += 10
                    elif card.id == Hariyama:
                        if field_counts[Makuhita] >= 1:
                            score += 20
                        else:
                            score -= 20
                    elif card.id == Lunatone:
                        if field_counts[card.id] >= 1:
                            score -= 250
                        else:
                            score += 60
                    elif card.id == Solrock:
                        if field_counts[card.id] >= 1:
                            score -= 250
                        else:
                            score += 50
                    elif card.id == Riolu:
                        if field_counts[card.id] + field_counts[Mega_Lucario_ex] >= 2:
                            score -= 150
                        elif field_counts[card.id] + field_counts[Mega_Lucario_ex] >= 1:
                            score -= 3
                        else:
                            score += 40
                    elif card.id == Mega_Lucario_ex:
                        if field_counts[Riolu] >= 1:
                            score += 40
                        else:
                            score -= 15
                    elif card.id == Basic_Fighting_Energy:
                        if not ability_used or not state.energyAttached:
                            score += 30
                        else:
                            score -= 1
                elif context == SelectContext.ATTACH_FROM:
                    score = energy_score(card, o.area == AreaType.ACTIVE)
        elif o.type == OptionType.PLAY:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            data = card_table[card.id]
            if data.cardType == CardType.POKEMON:
                score = 20000
                if card.id == Lunatone or card.id == Solrock:
                    if field_counts[card.id] >= 1:
                        score = -1
                elif card.id == Riolu:
                    if field_counts[card.id] + field_counts[Mega_Lucario_ex] >= 2:
                        score = -1
            else:
                score = 10000
                if card.id == Switch:
                    if plan.attacker <= 0:
                        score = -1
                    else:
                        score = 6000
                elif card.id == Premium_Power_Pro:
                    if state.supporterPlayed and plan.remain_hp <= 0:
                        score = -1
                    elif not can_attack:
                        if not state.supporterPlayed and hand_counts[Carmine] > 0 and hand_counts[Lillie_Determination] == 0:
                            score = 3050
                        else:
                            score = -1
                    else:
                        score = 5000
                elif card.id == Boss_Orders:
                    if plan.target >= 1:
                        score = 3200
                    else:
                        score = -1
                elif card.id == Carmine:
                    score = 3000
                elif card.id == Lillie_Determination:
                    score = 3100
                elif card.id == Gravity_Mountain:
                    if stadium_id == 0:
                        score = -1
        elif o.type == OptionType.ATTACH:
            card = get_card(obs, AreaType.HAND, o.index, my_index)
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            if card.id == Hero_Cape:
                score = 7000
                if pokemon.id == Riolu:
                    score += 100
                elif pokemon.id == Mega_Lucario_ex:
                    score += 200
            else:
                score = energy_score(pokemon, o.inPlayArea == AreaType.ACTIVE)
                if o.inPlayArea == AreaType.ACTIVE:
                    if plan.attacker == 0 and plan.energy:
                        score += 200
                else:
                    if plan.attacker == 1 + o.inPlayIndex and plan.energy:
                        score += 200
        elif o.type == OptionType.EVOLVE:
            pokemon = get_card(obs, o.inPlayArea, o.inPlayIndex, my_index)
            score = 9000 + len(pokemon.energies)
            if pokemon.id == Makuhita and plan.target == 0:
                score = -1
        elif o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if card.id == 1267:  # Lumiose City
                score = 1
            else:
                score = 30000
        elif o.type == OptionType.RETREAT:
            if plan.attacker >= 1:
                score = 2000
            else:
                score = -1
        elif o.type == OptionType.ATTACK:
            score = 1000
            if plan.attack_index == 1:
                if o.attackId == 983:  # Mega Brave
                    score += 100
            else:
                if o.attackId != 983:
                    score += 100

        scores.append(score)

    # Select in descending order of score
    desc_indices = [i for i, _ in sorted(enumerate(scores), key=lambda x: x[1], reverse=True)]
    if context == SelectContext.MAIN:
        o = select.option[desc_indices[0]]
        if o.type == OptionType.ABILITY:
            card = get_card(obs, o.area, o.index, my_index)
            if card.id == Lunatone:
                ability_used = True
    return desc_indices[:select.maxCount]

Overwriting deck2.py


# It's time to d d d d d duel

In [43]:
!python battle_simulator.py \
  --agent-a deck1.py \
  --agent-b deck2.py \
  --deck-a /kaggle/input/datasets/kiyotah/dragapult-ex-deck/deck.csv \
  --deck-b /kaggle/input/datasets/kiyotah/mega-lucario-ex-deck/deck.csv \
  --matches 1000 \
  --alternate-seats \
  --on-invalid-selection loss

[load] importing agent A: deck1.py
[deck] A: staging deck.csv from explicit deck file /kaggle/input/datasets/kiyotah/dragapult-ex-deck/deck.csv
[load] importing agent B: deck2.py
[deck] B: staging deck.csv from explicit deck file /kaggle/input/datasets/kiyotah/mega-lucario-ex-deck/deck.csv
[deck] A: 60 cards from /kaggle/working/deck1.py
[deck] B: 60 cards from /kaggle/working/deck2.py
[match 100/1000] winner=A steps=116 reason=player 1 has zero prize cards invalid=(0,0)
[match 200/1000] winner=B steps=163 reason=player 0 has zero prize cards invalid=(0,0)
[match 300/1000] winner=A steps=173 reason=player 1 has zero prize cards invalid=(0,0)
[match 400/1000] winner=B steps=156 reason=player 0 has zero prize cards invalid=(0,0)
[match 500/1000] winner=None steps=184 reason=forced_prompt_has_no_options:minCount=1 invalid=(0,0)
[match 600/1000] winner=A steps=147 reason=player 1 has zero prize cards invalid=(0,0)
[match 700/1000] winner=A steps=203 reason=player 1 has zero prize cards inv

## You can study erroneous game logs by using the --debug flag or looking into cabt_traces. For example:

```
python battle_simulator.py \
  --agent-a deck1.py \
  --agent-b deck2.py \
  --deck-a /kaggle/input/datasets/kiyotah/dragapult-ex-deck/deck.csv \
  --deck-b /kaggle/input/datasets/kiyotah/mega-lucario-ex-deck/deck.csv \
  --matches 1 \
  --alternate-seats \
  --debug \
  --abort-on-exception \
  --trace-dir cabt_debug_one
```